# Multi-Segment Streams

**Learning outcome:** Apply multi-segment streams through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Intermediate  
**Execution profile:** `base`  
**Expected runtime:** under 1 minute  
**Optional extras:** none

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** How should a stream with changing heat-capacity flow be represented without hiding its temperature intervals?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Define and target a segmented stream

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

segmented_input = {
    "streams": [
        {"zone": "Site", 
         "name": "Variable CP hot", 
         "segments": [
            {"name": "hot-1", "t_supply": 200.0, "t_target": 150.0, "heat_flow": 50.0},
            {"name": "hot-2", "t_supply": 150.0, "t_target": 100.0, "heat_flow": 100.0},
        ]},
        {
            "zone": "Site",
            "name": "Cold demand",
            "t_supply": 40.0,
            "t_target": 170.0,
            "heat_flow": 130.0,
        },
    ],
    "utilities": [],
}
problem = PinchProblem(segmented_input, project_name="Site")
validated = problem.validate()
target = problem.target.direct_heat_integration()
site_zone = problem.master_zone
problem.summary_frame()

## Step 2: Audit the prepared segments

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
prepared_hot = list(problem.hot_streams)[0]
prepared_cold = list(problem.cold_streams)[0]
hot_utilities = list(problem.hot_utilities)
cold_utilities = list(problem.cold_utilities)
segment_table = [
    {"name": segment.name, "supply": float(segment.supply_temperature), "target": float(segment.target_temperature), "duty": float(segment.heat_flow)}
    for segment in prepared_hot.segments
]
assert all(
    first.target_temperature == second.supply_temperature
    for first, second in zip(
        prepared_hot.segments, prepared_hot.segments[1:]
    )
)
composite = problem.plot.composite_curve()
segment_table

## Review the result

Inspect the prepared segment boundaries and duties beside the target summary before replacing the example with plant-specific variable-heat-capacity data.

In [ ]:
from IPython.display import display

display(segment_table)
display(problem.summary_frame())
display(composite)

## Interpret the result

Confirm adjacent segments are continuous and inspect each segment's temperature range and duty before trusting the target.

## Adapt this template

Add one segment per thermodynamic interval; do not average heat-capacity flow across phase or property changes.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.